In [4]:
# 1-ЕСЕП
# Берілген 4x4 кіріс матрицасы мен 2x2 kernel арқылы feature map табу

import numpy as np

# Кіріс матрицасы
I = np.array([
    [1, 2, 0, 1],
    [0, 1, 2, 1],
    [1, 0, 1, 2],
    [0, 1, 0, 1]
])

# Kernel
K = np.array([
    [1, 0],
    [0, -1]
])

# CNN-дегі әдеттегі тәсіл: kernel-ді аудармай cross-correlation жасау
out_h = I.shape[0] - K.shape[0] + 1
out_w = I.shape[1] - K.shape[1] + 1

feature_map = np.zeros((out_h, out_w), dtype=int)

for i in range(out_h):
    for j in range(out_w):
        region = I[i:i+2, j:j+2]
        feature_map[i, j] = np.sum(region * K)

print("Кіріс I =")
print(I)

print("\nKernel K =")
print(K)

print("\nFeature map =")
print(feature_map)

Кіріс I =
[[1 2 0 1]
 [0 1 2 1]
 [1 0 1 2]
 [0 1 0 1]]

Kernel K =
[[ 1  0]
 [ 0 -1]]

Feature map =
[[ 0  0 -1]
 [ 0  0  0]
 [ 0  0  0]]


In [5]:
# 2-ЕСЕП
# 1-есептегі feature map нәтижесін ReLU арқылы өткізу

import numpy as np

feature_map = np.array([
    [ 0,  0, -1],
    [ 0,  0,  0],
    [ 0,  0,  0]
])

relu_output = np.maximum(0, feature_map)

print("Feature map =")
print(feature_map)

print("\nReLU нәтижесі =")
print(relu_output)


Feature map =
[[ 0  0 -1]
 [ 0  0  0]
 [ 0  0  0]]

ReLU нәтижесі =
[[0 0 0]
 [0 0 0]
 [0 0 0]]


In [6]:
# 3-ЕСЕП
# 4x4 feature map-ке 2x2 Max Pooling, stride=2 қолдану

import numpy as np

F = np.array([
    [1, 3, 2, 0],
    [4, 6, 5, 1],
    [0, 2, 1, 3],
    [1, 0, 2, 4]
])

pool_size = 2
stride = 2

out_h = (F.shape[0] - pool_size) // stride + 1
out_w = (F.shape[1] - pool_size) // stride + 1

pooled = np.zeros((out_h, out_w), dtype=int)

for i in range(out_h):
    for j in range(out_w):
        region = F[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
        pooled[i, j] = np.max(region)

print("Бастапқы feature map F =")
print(F)

print("\nMax Pooling нәтижесі =")
print(pooled)


Бастапқы feature map F =
[[1 3 2 0]
 [4 6 5 1]
 [0 2 1 3]
 [1 0 2 4]]

Max Pooling нәтижесі =
[[6 5]
 [2 4]]


In [7]:
# 4-ЕСЕП
# Берілген feature map-ке Dropout(rate=0.5) қолдану

import numpy as np

F = np.array([
    [2, 0],
    [3, 1]
], dtype=float)

dropout_rate = 0.5
keep_prob = 1 - dropout_rate

# Нәтиже қайталануы үшін seed
np.random.seed(42)

# Mask генерациялау
mask = np.random.binomial(1, keep_prob, size=F.shape)

# Қарапайым dropout
dropped = F * mask

# Inverted dropout керек болса:
inverted_dropped = (F * mask) / keep_prob

print("Feature map F =")
print(F)

print("\nDropout mask =")
print(mask)

print("\nЖаңа feature map (қарапайым dropout) =")
print(dropped)

print("\nЖаңа feature map (inverted dropout) =")
print(inverted_dropped)

Feature map F =
[[2. 0.]
 [3. 1.]]

Dropout mask =
[[0 1]
 [1 1]]

Жаңа feature map (қарапайым dropout) =
[[0. 0.]
 [3. 1.]]

Жаңа feature map (inverted dropout) =
[[0. 0.]
 [6. 2.]]


In [8]:
# 5-ЕСЕП
# X = [2, 4, 6, 8] векторына Batch Normalization қолдану, epsilon = 0.01

import numpy as np

X = np.array([2, 4, 6, 8], dtype=float)
eps = 0.01

mu = np.mean(X)
var = np.var(X)   # population variance
x_hat = (X - mu) / np.sqrt(var + eps)

print("X =", X)
print("mean(X) =", mu)
print("var(X) =", var)
print("epsilon =", eps)
print("\nNormalized X_hat =")
print(x_hat)


X = [2. 4. 6. 8.]
mean(X) = 5.0
var(X) = 5.0
epsilon = 0.01

Normalized X_hat =
[-1.34030115 -0.44676705  0.44676705  1.34030115]


In [9]:

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

# 1. CIFAR-10 жүктеу
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)

# 2. ResNet50 үшін суреттерді 224x224 өлшеміне келтіру
IMG_SIZE = 224
NUM_CLASSES = 10
BATCH_SIZE = 32

def preprocess_image(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)   # ResNet50-ге арналған preprocessing
    label = tf.squeeze(label)
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.map(preprocess_image).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = test_ds.map(preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# 3. Алдын ала оқытылған ResNet50 негізін жүктеу
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 4. Негізгі қабаттарды freeze жасау
base_model.trainable = False

# 5. Жаңа классификатор қабатын қосу
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

# 6. Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 7. Summary
model.summary()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 296s 2us/step
Train: (50000, 32, 32, 3) (50000, 1)
Test : (10000, 32, 32, 3) (10000, 1)
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        20,490 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,608,202 (90.06 MB)

 Trainable params: 20,490 (80.04 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [10]:
# 7-ЕСЕП
# 28x28 суретке:
# 3x3 kernel -> ReLU -> MaxPooling 2x2 -> Dropout(rate=0.25)
# Соңғы feature map өлшемін есептеу

import numpy as np

# 1 арналы 28x28 сурет
np.random.seed(1)
image = np.random.randint(0, 256, size=(28, 28)).astype(float)

# 3x3 kernel
kernel = np.array([
    [ 1,  0, -1],
    [ 1,  0, -1],
    [ 1,  0, -1]
], dtype=float)

# ===== Convolution (valid) =====
conv_h = image.shape[0] - kernel.shape[0] + 1   # 26
conv_w = image.shape[1] - kernel.shape[1] + 1   # 26

conv_out = np.zeros((conv_h, conv_w))

for i in range(conv_h):
    for j in range(conv_w):
        region = image[i:i+3, j:j+3]
        conv_out[i, j] = np.sum(region * kernel)

# ===== ReLU =====
relu_out = np.maximum(0, conv_out)

# ===== Max Pooling 2x2, stride=2 =====
pool_size = 2
stride = 2

pool_h = (relu_out.shape[0] - pool_size) // stride + 1   # 13
pool_w = (relu_out.shape[1] - pool_size) // stride + 1   # 13

pool_out = np.zeros((pool_h, pool_w))

for i in range(pool_h):
    for j in range(pool_w):
        region = relu_out[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
        pool_out[i, j] = np.max(region)

# ===== Dropout =====
dropout_rate = 0.25
keep_prob = 1 - dropout_rate
mask = np.random.binomial(1, keep_prob, size=pool_out.shape)
dropout_out = pool_out * mask

print("Бастапқы сурет өлшемі:", image.shape)
print("Convolution-нан кейін:", conv_out.shape)
print("ReLU-дан кейін:", relu_out.shape)
print("MaxPooling-нен кейін:", pool_out.shape)
print("Dropout-тан кейін:", dropout_out.shape)

print("\nСоңғы feature map өлшемі =", dropout_out.shape)

Бастапқы сурет өлшемі: (28, 28)
Convolution-нан кейін: (26, 26)
ReLU-дан кейін: (26, 26)
MaxPooling-нен кейін: (13, 13)
Dropout-тан кейін: (13, 13)

Соңғы feature map өлшемі = (13, 13)


## Нәтижелерді интерпретациялау

### 1-ЕСЕП: Convolution (Бүрмелеу) қабаты
*   **Мақсаты:** Берілген кіріс матрицасынан және 2x2 ядродан (kernel) feature map (ерекшелік картасы) алу.
*   **Нәтиже:** Кіріс `I` матрицасы мен `K` ядросы арқылы 3x3 өлшемді `feature_map` алынды.
*   **Түсіндірме:** Бұл есеп Convolutional Neural Networks (CNN) негізгі операциясы болып табылатын cross-correlation процесін көрсетеді. Ядро кіріс матрицасының үстінен жылжып, әр позицияда элементтерді көбейтіп, қосу арқылы жаңа мәндерді есептейді. Нәтижесінде, 4x4 кіріс матрицасынан 2x2 ядроны қолдану арқылы 3x3 feature map алынды.

### 2-ЕСЕП: ReLU (Rectified Linear Unit) активациясы
*   **Мақсаты:** 1-есептегі feature map нәтижесін ReLU функциясынан өткізу арқылы теріс мәндерді нөлге айналдыру.
*   **Нәтиже:** `feature_map` теріс мәндері (`-1`) нөлге айналып, `relu_output` матрицасы алынды.
*   **Түсіндірме:** ReLU — бұл нейрондық желілерде жиі қолданылатын активация функциясы. Ол желіге сызықтық емес қабілет береді, теріс мәндерді нөлге айналдырады және оң мәндерді өзгеріссіз қалдырады. Бұл желінің оқу процесін жеделдетеді және "vanishing gradient" мәселесін шешуге көмектеседі.

### 3-ЕСЕП: Max Pooling қабаты
*   **Мақсаты:** Feature map-тың өлшемін кішірейту және ең маңызды ақпаратты сақтау.
*   **Нәтиже:** 4x4 `F` feature map-тен 2x2 `pooled` матрицасы алынды. `stride=2` болғандықтан, әр 2x2 аймақтың ең үлкен мәні таңдалып, келесі қабатқа берілді.
*   **Түсіндірме:** Max Pooling — бұл ішкі іріктеу (downsampling) операциясы. Ол кіріс матрицадағы (feature map) әр аймақтан (мысалы, 2x2 немесе 3x3) ең үлкен мәнді таңдап алады. Бұл әдіс желінің есептеу жүктемесін азайтады, артық үйренуді (overfitting) азайтады және суреттегі нысандардың орналасуындағы аздаған өзгерістерге (translation invariance) тұрақтылық береді.

### 4-ЕСЕП: Dropout қабаты
*   **Мақсаты:** Нейрондық желіде артық үйренуді (overfitting) азайту.
*   **Нәтиже:** Бастапқы `F` feature map-ке `dropout_rate=0.5` қолданылып, кездейсоқ нейрондар нөлге айналдырылған `dropped` және `inverted_dropped` матрицалары алынды.
*   **Түсіндірме:** Dropout — бұл желінің артық үйренуін болдырмау үшін қолданылатын регуляризация әдісі. Ол әрбір оқыту итерациясында желідегі нейрондардың белгілі бір пайызын кездейсоқ түрде өшіреді (олардың шығыстарын нөлге теңестіреді). Бұл әдіс әрбір нейронның басқа нейрондарға "сенім артуын" азайтады, желіні неғұрлым мықты және жалпылайтын етеді. Inverted dropout әдісі кезінде, оқыту кезінде қалған белсенді нейрондардың шығыстарын `1 / (1 - dropout_rate)` мөлшеріне көбейту арқылы тест кезінде масштабтау қажеттілігі жойылады.

### 5-ЕСЕП: Batch Normalization
*   **Мақсаты:** Нейрондық желідегі әр қабаттың кірісін стандарттау арқылы оқытуды тұрақтандыру және жеделдету.
*   **Нәтиже:** `X` векторының орташа мәні мен дисперсиясы есептеліп, `epsilon` қосу арқылы `x_hat` нормаланған векторы алынды.
*   **Түсіндірме:** Batch Normalization — бұл желінің оқыту процесін жылдамдататын және тұрақтандыратын әдіс. Ол әр мини-пакеттегі (mini-batch) кірістерді орташа мәні нөлге, ал дисперсиясы бірге тең болатындай етіп нормалайды. Бұл "internal covariate shift" мәселесін азайтады, яғни алдыңғы қабаттардың параметрлерінің өзгеруіне байланысты келесі қабаттардың кіріс бөлуінің өзгеруін азайтады. Нәтижесінде, жоғары оқыту жылдамдығын қолдануға болады және модельдің өнімділігі жақсарады.

### 6-ЕСЕП: ResNet50 Transfer Learning моделін құру
*   **Мақсаты:** CIFAR-10 деректер жиынтығын жіктеу үшін алдын ала оқытылған ResNet50 моделін қолдана отырып, жаңа нейрондық желі моделін құру.
*   **Нәтиже:** CIFAR-10 деректер жиынтығы жүктеліп, ResNet50 моделі үшін суреттер 224x224 өлшеміне келтірілді, алдын ала оқытылған ResNet50 негізі жүктеліп, оның қабаттары мұздатылды (`trainable=False`). Оның үстіне GlobalAveragePooling2D, Dropout (0.3) және 10 шығыс сыныбы бар Dense қабаты қосылып, модель құрылып, компиляцияланды.
*   **Түсіндірме:** Бұл есеп Transfer Learning (білімді тасымалдау) принципін көрсетеді. ResNet50 сияқты үлкен модельдер (ImageNet сияқты үлкен деректер жиынтығында оқытылған) суреттерден жалпы ерекшеліктерді (features) тануға қабілетті. Бұл модельді кішігірім деректер жиынтығында (CIFAR-10) қолдану үшін, оның негізгі қабаттарын мұздатып, үстіне жаңа, кішігірім классификатор қабаттарын (Dense, Dropout) қосу арқылы модельді белгілі бір тапсырмаға бейімдеуге болады. Бұл әдіс модельді нөлден оқытуға қарағанда әлдеқайда жылдам және тиімді, әсіресе деректер аз болған жағдайда.

### 7-ЕСЕП: CNN қабаттарының тізбегі арқылы Feature Map өлшемін есептеу
*   **Мақсаты:** 28x28 суретке Convolution, ReLU, Max Pooling және Dropout қабаттарын қолданғаннан кейінгі соңғы feature map өлшемін анықтау.
*   **Нәтиже:** Бастапқы 28x28 сурет 3x3 kernel-мен Convolution (valid) арқылы 26x26-ға айналды, ReLU бұл өлшемді өзгертпеді, 2x2 Max Pooling (stride=2) оны 13x13-ке дейін кішірейтті, ал Dropout өлшемді өзгертпестен 13x13 feature map-ті қалдырды.
*   **Түсіндірме:** Бұл есеп CNN архитектурасындағы әртүрлі қабаттардың feature map өлшеміне қалай әсер ететінін көрсетеді:
    *   **Convolution (valid):** Output size = (Input size - Filter size + 1). (28 - 3 + 1) = 26.
    *   **ReLU:** Өлшемді өзгертпейді.
    *   **Max Pooling:** Output size = ((Input size - Pool size) / Stride) + 1. ((26 - 2) / 2) + 1 = 13.
    *   **Dropout:** Өлшемді өзгертпейді, тек мәндерді кездейсоқ түрде нөлге айналдырады.
    Соңғы нәтиже 13x13 feature map болып табылады.